<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B01%5D%20-%20Intro_No_Supervisados/%5B01%5D%20-%20Notebooks/E3_Perfila_e_Interpreta_tus_Grupos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E3 · Perfila e interpreta tus grupos - Introducción a los modelos no supervisados

## Introducción

> El modelo agrupa y **tú interpretas**. K-Means devuelve grupos sin nombre; sin interpretar,
> no hay valor.

En este ejercicio hacemos **profiling**: describir cada cluster con métricas simples,
compararlo con el resto, buscar un **cliente real representativo** (la idea de **K-Medoids**:
el medoide es un dato real, no un promedio inventado), ponerle un **nombre accionable** y
convertirlo en una **decisión** (una campaña).

## Objetivos del ejercicio

- Calcular el **perfil medio** de cada cluster.
- Comparar cada grupo **frente al resto** para ver qué lo hace especial.
- Encontrar el **medoide** (cliente real representativo) de cada grupo.
- Ponerle **nombre** y asignarle una **campaña**.

## Descripción del dataset (clientes sin etiqueta)

Imagina que tienes una base de clientes y quieres ofrecerles un descuento, pero **no hay
etiquetas**: nadie te ha dicho qué cliente es de qué tipo. El objetivo del aprendizaje no
supervisado es justo ese: **descubrir la estructura** que hay dentro de los datos.

Generamos un dataset **sintético y reproducible** con `generar_clientes` (autocontenido en
Colab). Cada fila es un cliente con estas variables:

| Variable | Tipo | Descripción |
|---|---|---|
| `gasto_anual` | numérica | Gasto total al año (€), escala de miles |
| `num_visitas` | numérica | Nº de visitas al año, escala de decenas |
| `ticket_medio` | numérica | Gasto medio por compra (€) |
| `antiguedad_meses` | numérica | Meses como cliente |
| `edad` | numérica | Edad del cliente |
| `usa_app` | binaria | 1 si usa la app |
| `tiene_tarjeta_fidelidad` | binaria | 1 si tiene tarjeta de fidelidad |
| `compra_online` | binaria | 1 si compra online |
| `recibe_newsletter` | binaria | 1 si recibe la newsletter |
| `devuelve_productos` | binaria | 1 si suele devolver productos |

> Fíjate en las **escalas tan distintas** (gasto en miles, visitas en decenas). Esto va a ser
> clave: en clustering, la distancia depende de la escala, así que **habrá que escalar**.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### 2. Datos, escalado y clustering

In [ ]:
import numpy as np
import pandas as pd

def generar_clientes(n=600, semilla=42):
    # Dataset sintetico y reproducible de clientes SIN ETIQUETA para segmentacion.
    # Por dentro hay 4 perfiles latentes que el modelo deberia redescubrir, pero NO los
    # exponemos: en aprendizaje no supervisado no hay target, solo buscamos estructura.
    rng = np.random.default_rng(semilla)
    # perfil: (gasto_anual, num_visitas, ticket_medio, antiguedad_meses, edad,
    #          p_app, p_fidelidad, p_online, p_newsletter, p_devuelve)
    perfiles = [
        (9000, 42, 230, 60, 46, 0.85, 0.90, 0.70, 0.60, 0.10),  # grandes clientes
        (1100,  6, 120, 22, 37, 0.40, 0.20, 0.55, 0.30, 0.10),  # ocasionales
        (3200, 36,  75, 44, 52, 0.50, 0.65, 0.60, 0.80, 0.55),  # cazaofertas
        (2400, 15, 165,  9, 30, 0.92, 0.40, 0.95, 0.50, 0.20),  # nuevos digitales
    ]
    pesos = [0.22, 0.33, 0.25, 0.20]
    seg = rng.choice(len(perfiles), size=n, p=pesos)

    filas = []
    for s in seg:
        g, v, t, a, e, pa, pf, po, pn, pdv = perfiles[s]
        filas.append([
            round(max(50, rng.normal(g, g * 0.22)), 2),    # gasto_anual (€)
            int(max(1, round(rng.normal(v, v * 0.30)))),    # num_visitas
            round(max(5, rng.normal(t, t * 0.22)), 2),      # ticket_medio (€)
            int(max(1, round(rng.normal(a, 12)))),          # antiguedad_meses
            int(np.clip(rng.normal(e, 8), 18, 85)),         # edad
            int(rng.random() < pa),                          # usa_app
            int(rng.random() < pf),                          # tiene_tarjeta_fidelidad
            int(rng.random() < po),                          # compra_online
            int(rng.random() < pn),                          # recibe_newsletter
            int(rng.random() < pdv),                         # devuelve_productos
        ])
    cols = ["gasto_anual", "num_visitas", "ticket_medio", "antiguedad_meses", "edad",
            "usa_app", "tiene_tarjeta_fidelidad", "compra_online", "recibe_newsletter",
            "devuelve_productos"]
    return pd.DataFrame(filas, columns=cols)

In [ ]:
df = generar_clientes(n=600, semilla=42)
X = df.to_numpy(dtype=float)
X_esc = StandardScaler().fit_transform(X)

# Elegimos K por silhouette (como en E2) y entrenamos el modelo final
mejor_k, mejor_sil = None, -1
for k in range(2, 7):
    lab = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=0).fit_predict(X_esc)
    s = silhouette_score(X_esc, lab)
    if s > mejor_sil:
        mejor_k, mejor_sil = k, s

km = KMeans(n_clusters=mejor_k, init="k-means++", n_init=10, random_state=0)
df["cluster"] = km.fit_predict(X_esc)
print(f"K elegido: {mejor_k} (silhouette {mejor_sil:.3f})")
print("Tamaño de cada cluster:", np.bincount(df["cluster"]))

### 3. Perfil medio de cada cluster

In [ ]:
perfil = df.groupby("cluster").mean().round(1)
perfil

### 4. Comparar cada cluster frente al resto

Para ver qué hace especial a cada grupo, comparamos su media con la media global (un valor
> 1 significa "por encima de la media", < 1 "por debajo").

In [ ]:
media_global = df.drop(columns="cluster").mean()
ratio = (df.groupby("cluster").mean() / media_global).round(2)
print("Cada celda: cuántas veces la media del grupo respecto a la media global")
ratio

### 5. El medoide: un cliente real representativo

El **centroide** de K-Means es un punto promedio que no existe. Si quieres un **cliente real**
que represente al grupo (la idea de **K-Medoids**), tomamos la observación real más cercana al
centroide de su cluster.

In [ ]:
medoides = []
for c in range(mejor_k):
    idx = np.where(km.labels_ == c)[0]
    d = ((X_esc[idx] - km.cluster_centers_[c]) ** 2).sum(axis=1)
    medoides.append(idx[d.argmin()])

print("Cliente real más representativo de cada cluster (medoide):")
df.iloc[medoides].assign(cluster=range(mejor_k)).set_index("cluster")

### 6. Pon nombre y decide la campaña

Con el perfil delante, traducimos cada cluster a algo accionable. (Ajusta los nombres y
campañas según los números que te hayan salido arriba.)

In [ ]:
# Ordenamos los clusters por gasto medio para describirlos de mayor a menor valor
orden = df.groupby("cluster")["gasto_anual"].mean().sort_values(ascending=False)
print("Gasto medio por cluster (de mayor a menor):")
print(orden.round(0))

resumen = pd.DataFrame({
    "cluster": orden.index,
    "gasto_medio": orden.round(0).values,
    "n_clientes": [int((df["cluster"] == c).sum()) for c in orden.index],
})
# Etiquetas de ejemplo segun el ranking de gasto (revisa que encajen con tu perfil)
nombres = ["VIP / grandes clientes", "Cliente medio", "Cazaofertas", "Ocasional / dormido",
           "Extra 1", "Extra 2"]
campanas = ["Programa premium y atención prioritaria",
            "Venta cruzada para subir ticket",
            "Ofertas y packs de descuento",
            "Campaña de reactivación",
            "-", "-"]
resumen["nombre_sugerido"] = nombres[:len(resumen)]
resumen["campaña_sugerida"] = campanas[:len(resumen)]
resumen

### Reflexión

1. ¿Qué variable distingue más a cada cluster del resto (mira la tabla de ratios)?
2. ¿En qué se parece y en qué se diferencia el medoide del centroide del mismo grupo?
3. ¿Los nombres que has puesto son accionables (llevan a una decisión clara)?
4. Si tuvieras que lanzar UNA sola campaña, ¿a qué cluster se la harías y por qué?